## Python Replay Notebook

This notebook replays historical candles one bar at a time using the same VWAP probability band engine used by the live workflow.

It is not connected to TradingView. TradingView replay will require a separate Pine Script overlay later.

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("✅ Project root added to Python path:")
print(PROJECT_ROOT)

In [ ]:
from src.config import CONFIG
from src.loaders import load_tradingview_csv, assign_sessions
from src.engine import EngineState, update_engine_state
from src.replay import run_replay

### Load historical data

This loads the configured historical candle file and assigns session ids before replay.

In [ ]:
csv_path = CONFIG["data_dir"] / CONFIG["csv_filename"]

if not csv_path.is_absolute():
    csv_path = PROJECT_ROOT / csv_path

print("CSV path:", csv_path)
print("Exists:", csv_path.exists())

df = load_tradingview_csv(csv_path)
df = assign_sessions(df, CONFIG["sessions"])

print("Rows:", len(df))
display(df.head())
display(df.tail())

### Load probability tables

The replay engine needs a calibrated probability table. The marginal table is used as a fallback when exact trend/context bins are sparse.

In [ ]:
# ── Demo: step through 10 bars of replay and print state ──
print("Replay demo (first 10 bars):")
print(f"{'Bar':>4} {'Zone':>5} {'Z-Score':>8} {'P(MR)':>7} {'P(CONT)':>8} {'Trend':>8}")
print("-" * 50)

for i, state in enumerate(
    run_replay(
        df,
        CONFIG,
        prob_table,
        EngineState=EngineState,
        update_engine_state=update_engine_state,
        speed=1e9,
    )
):
    if i >= 10:
        break
    mr = state.probabilities.get('MR', {}).get('prob', 0)
    cont = state.probabilities.get('CONT', {}).get('prob', 0)
    trend = state.context.get('trend_bin', '?')
    print(f"{i:>4} {state.zone:>5} {state.z_score:>8.3f} {mr:>7.1%} {cont:>8.1%} {trend:>8}")